# 2016~2020·2022~2024 TF-IDF 예측 대상 데이터 통합

2021년 수작업 라벨은 학습 기준으로 분리하고, 나머지 8개 연도×17개 시도 정제 데이터를 TF-IDF 예측 대상으로 취합한다.

- 데이터 스냅샷: 2026-07-28 지역별 정제 CSV 136개
- 모델 학습·평가: 미수행
- 데이터 분할 전략·평가 지표·CV fold 수: N/A

In [ ]:
from pathlib import Path
import sys

import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.consolidate_tfidf_prediction_targets import (  # noqa: E402
    PREDICTION_YEARS,
    consolidate_prediction_targets,
    save_outputs,
)

np.random.seed(42)
SOURCE_DIR = PROJECT_ROOT / "data/interim"
OUTPUT_DIR = SOURCE_DIR / "영역분류_라벨링/예측대상"

In [ ]:
combined, qa = consolidate_prediction_targets(SOURCE_DIR)
paths = save_outputs(combined, qa, OUTPUT_DIR)
qa

In [ ]:
year_summary = combined.groupby("연도", sort=True).agg(
    지역수=("지역", "nunique"),
    행수=("원본행", "size"),
    주요내용_정제_결측=("주요내용_정제_결측", "sum"),
)
year_summary

In [ ]:
assert set(combined["연도"]) == set(PREDICTION_YEARS)
assert 2021 not in set(combined["연도"])
assert combined["지역"].nunique() == 17
assert len(qa) == 8 * 17
assert len(combined) == 50678
assert not combined.duplicated(["연도", "지역", "원본행"]).any()
assert int(combined["주요내용_정제_결측"].sum()) == 363
assert not combined["분류텍스트"].eq("").any()
paths